In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("esquemagold", "gold")
dbutils.widgets.text("catalogo", "etl_vinkos")


In [0]:
esquemagold = dbutils.widgets.get("esquemagold")
catalogo = dbutils.widgets.get("catalogo")



In [0]:

#Extraemos la tabla de Estadisticos 

from pyspark.sql.functions import col

df_estadistica = spark.table(
    f"{catalogo}.{esquemagold}.estadistica"
)

display(df_estadistica)

In [0]:
#Crear el DataFrame agrupado por email

from pyspark.sql.functions import *

df_visitantes = (
    df_estadistica
    .groupBy("email")
    .agg(

        # Primera visita
       min("fecha_click").alias("fechaPrimeraVisita"),

        # Última visita
        max("fecha_click").alias("fechaUltimaVisita"),

        # Total de visitas
        count("*").alias("visitasTotales"),

        # Visitas del año actual
        sum(
            when(
                year(col("fecha_click")) == year(current_timestamp()),
                1
            ).otherwise(0)
        ).alias("visitasAnioActual"),

        # Visitas del mes actual
        sum(
            when(
                (year(col("fecha_click")) == year(current_timestamp())) &
                (month(col("fecha_click")) == month(current_timestamp())),
                1
            ).otherwise(0)
        ).alias("visitasMesActual")

    )
    .withColumn(
        "fechaCarga",
        current_timestamp()
    )
    .withColumn(
        "fechaActualizacion",
        current_timestamp()
    )
)

display(df_visitantes)


In [0]:
#Crea un MERGE para operaciones (UPSERT). actualiza e inserta en una misma instruccion dependiendo del caso 

#Crea vista
df_visitantes.createOrReplaceTempView("vw_visitantes")


display(spark.sql("SELECT * FROM vw_visitantes"))

In [0]:
%sql

MERGE INTO ${catalogo}.${esquemagold}.visitantes AS T
USING vw_visitantes AS S

ON T.email = S.email

WHEN MATCHED THEN
UPDATE SET

    T.fechaPrimeraVisita = LEAST(
        T.fechaPrimeraVisita,
        S.fechaPrimeraVisita
    ),

    T.fechaUltimaVisita = GREATEST(
        T.fechaUltimaVisita,
        S.fechaUltimaVisita
    ),

    T.visitasTotales = T.visitasTotales + S.visitasTotales,

    T.visitasAnioActual = T.visitasAnioActual + S.visitasAnioActual,

    T.visitasMesActual = T.visitasMesActual + S.visitasMesActual,

    T.fechaActualizacion = current_timestamp()


WHEN NOT MATCHED THEN

INSERT (

    email,
    fechaPrimeraVisita,
    fechaUltimaVisita,
    visitasTotales,
    visitasAnioActual,
    visitasMesActual,
    fechaCarga,
    fechaActualizacion

)

VALUES (

    S.email,
    S.fechaPrimeraVisita,
    S.fechaUltimaVisita,
    S.visitasTotales,
    S.visitasAnioActual,
    S.visitasMesActual,
    current_timestamp(),
    current_timestamp()

);